# Panel Estimator Comparison: 2×2 Monte Carlo Study

Compares **DiD**, **Synthetic Control** (Doudchenko–Imbens elastic net), and **SDID** (Arkhangelsky et al., 2021) across a 2×2 DGP design that varies two independent axes:

| Knob | Low | High |
|------|-----|------|
| **σ_λ** (factor-loading dispersion) | 0 — additive, parallel trends exact | 0.5 — heterogeneous loadings, parallel trends violated |
| **T_pre / N_control** | 10 / 100 = 0.1 (T ≪ N) | 50 / 20 = 2.5 (T ≫ N) |

**DGP family:** $Y_{it} = \alpha_i + 0.3\,t + \lambda_i F_t + \varepsilon_{it}$, with $\alpha_i \sim \mathcal{N}(5, 2)$, $F_t \sim \mathcal{N}(0, 1)$, $\lambda_i \sim \mathcal{N}(1, \sigma_\lambda)$, $\varepsilon_{it} \sim \mathcal{N}(0, 0.1)$. True ATT calibrated to 2% of the treated unit's counterfactual outcome at treatment onset.

**Reproducibility:** Pre-computed Monte-Carlo results (100 seeds × 4 DGPs) are embedded as a Python literal; set `RUN_SIMULATION = True` to reproduce from scratch (~10–20 min). The simulation uses `panelib` — ensure it is importable from this notebook's directory.

## Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ── panelib ───────────────────────────────────────────────────────────────────
# Notebooks run with their directory as CWD, so a plain import suffices when
# panelib.py is in the same folder.  The sys.path guard catches edge cases.
_nb_dir = os.getcwd()
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

import panelib
from panelib import did, sc, sdid

print("Imports OK (panelib, numpy, pandas, matplotlib).")

## DGP Family & Monte Carlo Setup

The factor model nests both key axes:
- **σ_λ = 0**: λ_i = 1 for all units → common factor term β·F_t is absorbed into time FE → equivalent to additive model → parallel trends hold exactly
- **σ_λ = 0.5**: units respond differently to the common shock → parallel trends violated

The four DGP configurations:

| Label | σ_λ | T_pre | N_control | T/N |
|-------|-----|-------|-----------|-----|
| **Baseline** | 0 | 10 | 100 | 0.10 |
| **A: T-rich, additive** | 0 | 50 | 20 | 2.50 |
| **B: T-poor, factor** | 0.5 | 10 | 100 | 0.10 |
| **C: T-rich, factor** | 0.5 | 50 | 20 | 2.50 |

In [ ]:
# =============================================================================
# 1. DGP family
# =============================================================================
DGP_CONFIGS = {
    'Baseline':            dict(sigma_lambda=0.0, T_pre=10, N_control=100),
    'A: T-rich, additive': dict(sigma_lambda=0.0, T_pre=50, N_control=20),
    'B: T-poor, factor':   dict(sigma_lambda=0.5, T_pre=10, N_control=100),
    'C: T-rich, factor':   dict(sigma_lambda=0.5, T_pre=50, N_control=20),
}

T_POST   = 3
NOISE_SD = 0.1
ATT_PCT  = 0.02

DATA_DICT = {'treatment': 'treated', 'date': 'time',
             'post': 'post', 'unitid': 'unit_id', 'outcome': 'y'}


def make_panel_df(seed, T_pre, N_control, sigma_lambda):
    """Draw one panel from Y_it = alpha_i + 0.3 t + lambda_i F_t + eps_it."""
    np.random.seed(seed)
    N_total  = N_control + 1
    T_total  = T_pre + T_POST
    unit_ids = ['T000'] + [f'C{i:03d}' for i in range(N_control)]
    alpha_i  = np.random.normal(5, 2, N_total)
    lambda_i = np.random.normal(1, sigma_lambda, N_total)
    F_t      = np.random.normal(0, 1, T_total)
    rows, y_cf = [], None
    for i, uid in enumerate(unit_ids):
        is_treated = uid == 'T000'
        for t in range(T_total):
            y = alpha_i[i] + 0.3*t + lambda_i[i]*F_t[t] + np.random.normal(0, NOISE_SD)
            post = int(t >= T_pre)
            if is_treated and t == T_pre:
                y_cf = y
            rows.append({'unit_id': uid, 'time': t, 'y': y,
                         'treated': int(is_treated), 'post': post})
    true_att = ATT_PCT * y_cf
    df = pd.DataFrame(rows)
    df.loc[(df['treated'] == 1) & (df['post'] == 1), 'y'] += true_att
    return df, true_att


# =============================================================================
# 2. Estimators — direct panelib calls
#    sc.sc_att(fast=True) uses a single ElasticNetCV fit on the treated unit
#    with a coarser alpha grid — ~100× faster than LOO, same point estimate.
# =============================================================================
def _did(df):
    r = did.twfe(data=df, data_dict=DATA_DICT)
    return float(r['twfe']['coef_'].iloc[0])

def _sc(df):
    return sc.sc_att(model_name='di', data=df, data_dict=DATA_DICT, fast=True)

def _sdid(df):
    r = sdid.twfe_sdid(data=df, data_dict=DATA_DICT)
    return float(r['sdid'].loc['post_SDiD', 'coef_'])


# =============================================================================
# 3. Monte-Carlo runner + summary helper
# =============================================================================
def run_mc(T_pre, N_control, sigma_lambda, n_seeds=200):
    rows = []
    for seed in range(n_seeds):
        df, true_att = make_panel_df(seed, T_pre, N_control, sigma_lambda)
        row = {'seed': seed, 'true_att': true_att}
        for name, fn in [('did', _did), ('sc', _sc), ('sdid', _sdid)]:
            try:
                a = fn(df)
                row[f'{name}_att']  = a
                row[f'{name}_bias'] = a - true_att
            except Exception:
                row[f'{name}_att'] = row[f'{name}_bias'] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def summarise(res_df, label):
    out = []
    for model, col in [('DiD', 'did'), ('SC (DI)', 'sc'), ('SDID', 'sdid')]:
        b = res_df[f'{col}_bias'].dropna()
        q25, q75 = np.percentile(b, [25, 75])
        out.append({
            'DGP': label, 'Model': model, 'n': len(b),
            'True_ATT_mean': round(res_df['true_att'].mean(), 4),
            'Mean_ATT':      round(res_df[f'{col}_att'].dropna().mean(), 4),
            'Mean_bias':     round(b.mean(), 4),
            'Std':           round(b.std(),  4),
            'IQR_lo':        round(q25,      4),
            'IQR_hi':        round(q75,      4),
            'RMSE':          round(np.sqrt((b**2).mean()), 4),
        })
    return pd.DataFrame(out)


print("DGP and panelib-backed estimators defined (SC uses fast=True path).")

In [ ]:
# =============================================================================
# Run the simulation OR load pre-computed results
# =============================================================================
#  Set RUN_SIMULATION = True to re-run from scratch with the inline estimators
#  above. Expect ~10-20 min on a typical laptop (SDID is the bottleneck).
#  Default = False: load the embedded summary that produced the figures below.

RUN_SIMULATION = False
N_SEEDS        = 100

# Pre-computed results from a 100-seed × 4-DGP Monte Carlo (NOISE_SD=0.1).
# These are the canonical numbers referenced throughout the rest of the notebook.
EMBEDDED_SUMMARY = pd.DataFrame([
    {'DGP':'Baseline',            'Model':'DiD',    'n':100,'True_ATT_mean':0.1604,'Mean_ATT':0.1625,'Mean_bias': 0.0021,'Std':0.0580,'IQR_lo':-0.0405,'IQR_hi':0.0385,'RMSE':0.0578},
    {'DGP':'Baseline',            'Model':'SC (DI)','n':100,'True_ATT_mean':0.1604,'Mean_ATT':0.1974,'Mean_bias': 0.0370,'Std':0.1082,'IQR_lo':-0.0251,'IQR_hi':0.1004,'RMSE':0.1138},
    {'DGP':'Baseline',            'Model':'SDID',   'n':100,'True_ATT_mean':0.1604,'Mean_ATT':0.1591,'Mean_bias':-0.0013,'Std':0.0609,'IQR_lo':-0.0446,'IQR_hi':0.0357,'RMSE':0.0606},
    {'DGP':'A: T-rich, additive', 'Model':'DiD',    'n':100,'True_ATT_mean':0.4022,'Mean_ATT':0.3910,'Mean_bias':-0.0112,'Std':0.0586,'IQR_lo':-0.0525,'IQR_hi':0.0243,'RMSE':0.0593},
    {'DGP':'A: T-rich, additive', 'Model':'SC (DI)','n':100,'True_ATT_mean':0.4022,'Mean_ATT':0.4238,'Mean_bias': 0.0216,'Std':0.0842,'IQR_lo':-0.0334,'IQR_hi':0.0718,'RMSE':0.0865},
    {'DGP':'A: T-rich, additive', 'Model':'SDID',   'n':100,'True_ATT_mean':0.4022,'Mean_ATT':0.3963,'Mean_bias':-0.0058,'Std':0.0711,'IQR_lo':-0.0579,'IQR_hi':0.0443,'RMSE':0.0710},
    {'DGP':'B: T-poor, factor',   'Model':'DiD',    'n':100,'True_ATT_mean':0.1606,'Mean_ATT':0.2011,'Mean_bias': 0.0405,'Std':0.3522,'IQR_lo':-0.1381,'IQR_hi':0.1629,'RMSE':0.3528},
    {'DGP':'B: T-poor, factor',   'Model':'SC (DI)','n':100,'True_ATT_mean':0.1606,'Mean_ATT':0.2304,'Mean_bias': 0.0698,'Std':0.1496,'IQR_lo':-0.0314,'IQR_hi':0.1608,'RMSE':0.1644},
    {'DGP':'B: T-poor, factor',   'Model':'SDID',   'n':100,'True_ATT_mean':0.1606,'Mean_ATT':0.1566,'Mean_bias':-0.0040,'Std':0.0655,'IQR_lo':-0.0501,'IQR_hi':0.0310,'RMSE':0.0652},
    {'DGP':'C: T-rich, factor',   'Model':'DiD',    'n':100,'True_ATT_mean':0.4037,'Mean_ATT':0.4531,'Mean_bias': 0.0494,'Std':0.3827,'IQR_lo':-0.1096,'IQR_hi':0.1443,'RMSE':0.3839},
    {'DGP':'C: T-rich, factor',   'Model':'SC (DI)','n':100,'True_ATT_mean':0.4037,'Mean_ATT':0.4341,'Mean_bias': 0.0304,'Std':0.0911,'IQR_lo':-0.0316,'IQR_hi':0.0900,'RMSE':0.0956},
    {'DGP':'C: T-rich, factor',   'Model':'SDID',   'n':100,'True_ATT_mean':0.4037,'Mean_ATT':0.3979,'Mean_bias':-0.0058,'Std':0.0778,'IQR_lo':-0.0621,'IQR_hi':0.0460,'RMSE':0.0777},
])


if RUN_SIMULATION:
    import time
    print(f"Re-running Monte Carlo with {N_SEEDS} seeds per DGP...\n")
    summaries = []
    for name, cfg in DGP_CONFIGS.items():
        t0 = time.time()
        print(f"  {name}  (sigma_lambda={cfg['sigma_lambda']}, "
              f"T_pre={cfg['T_pre']}, N_ctrl={cfg['N_control']})")
        res = run_mc(**cfg, n_seeds=N_SEEDS)
        summaries.append(summarise(res, name))
        print(f"    done in {time.time()-t0:.0f}s")
        for m, c in [('DiD','did'),('SC','sc'),('SDID','sdid')]:
            b = res[f'{c}_bias'].dropna()
            print(f"    {m:5s}: bias={b.mean():+.4f}  rmse={np.sqrt((b**2).mean()):.4f}")
    summary_df = pd.concat(summaries, ignore_index=True)
    print("\nSimulation complete.")
else:
    summary_df = EMBEDDED_SUMMARY.copy()
    print(f"Loaded embedded results (100 seeds × 4 DGPs × 3 models = {len(summary_df)} rows)")
    print("Set RUN_SIMULATION = True above to reproduce from scratch.")

## Results

In [ ]:
from IPython.display import display

# ── Pivot: mean bias ──────────────────────────────────────────────────────────
pivot_bias = summary_df.pivot_table(
    index='DGP', columns='Model', values='Mean_bias'
)[['DiD', 'SC (DI)', 'SDID']]

# Reorder rows to match the 2×2 design
row_order = list(DGP_CONFIGS.keys())
pivot_bias = pivot_bias.reindex(row_order)

# ── Pivot: RMSE ───────────────────────────────────────────────────────────────
pivot_rmse = summary_df.pivot_table(
    index='DGP', columns='Model', values='RMSE'
)[['DiD', 'SC (DI)', 'SDID']].reindex(row_order)

print("=" * 60)
print("  MEAN BIAS  (100 seeds each)")
print("=" * 60)
display(pivot_bias.round(4))

print()
print("=" * 60)
print("  RMSE  = √(bias² + variance)")
print("=" * 60)
display(pivot_rmse.round(4))

print()
print("  Full detail:")
cols = ['DGP', 'Model', 'Mean_bias', 'Std', 'IQR_lo', 'IQR_hi', 'RMSE']
display(summary_df[cols].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dgp_labels   = list(DGP_CONFIGS.keys())
short_labels = ['Baseline', 'A: T-rich\nadditive', 'B: T-poor\nfactor', 'C: T-rich\nfactor']
models       = ['DiD', 'SC (DI)', 'SDID']
colors       = {'DiD': 'steelblue', 'SC (DI)': 'mediumpurple', 'SDID': 'coral'}
x            = np.arange(len(dgp_labels))
width        = 0.25

for ax, metric, title in zip(axes, ['Mean_bias', 'RMSE'], ['Mean Bias', 'RMSE']):
    for i, model in enumerate(models):
        vals = [
            summary_df.loc[(summary_df['DGP'] == d) & (summary_df['Model'] == model),
                           metric].values[0]
            for d in dgp_labels
        ]
        ax.bar(x + i * width, vals, width, label=model,
               color=colors[model], alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(x + width)
    ax.set_xticklabels(short_labels, fontsize=9)
    ax.set_ylabel(title)
    ax.set_title(f'{title} by DGP and Estimator')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('2×2 Monte Carlo: DiD vs SC vs SDID (100 seeds each)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Theoretical Analysis: Why SDID Is Robust Across All Four Cells

The treatment effects $\hat{\tau}$ we report below are functions of one random panel draw. All "expectation" statements that follow are **conditional on the realized DGP draw** (i.e. on $\{\alpha_i, \lambda_i, F_t, \varepsilon_{it}\}$) — they describe the algorithmic bias built into each estimator, not a Monte-Carlo average. The MC simply traces how that conditional bias propagates across draws.

### 1. The SDID Estimator

SDID is a doubly-weighted difference-in-differences:

$$\hat{\tau}^{SDID} = \underbrace{\left(\bar{Y}^{post}_{treat} - \sum_j \hat{\omega}_j \bar{Y}^{post}_j\right)}_{\text{post-period gap}} \;-\; \underbrace{\sum_{t < T_0} \hat{\lambda}_t \left(Y_{treat,t} - \sum_j \hat{\omega}_j Y_{j,t}\right)}_{\text{lambda-weighted pre-period gap}}$$

Both weight vectors live on a simplex with an intercept absorbed by least squares:

$$\hat{\omega} = \arg\min_{\omega \geq 0,\; \sum_j \omega_j = 1}\;\left\|Y^{pre}_{treat} - \omega_0 - Y^{pre}_{ctrl}\,\omega\right\|^2 + \zeta^2 T_{pre} \|\omega\|^2$$

$$\hat{\lambda} = \arg\min_{\lambda \geq 0,\; \sum_t \lambda_t = 1}\;\sum_{j} \left(\lambda_0 + \sum_{t < T_0} \lambda_t Y_{j,t} - \bar{Y}^{post}_{j}\right)^2$$

These two simplex constraints — $\sum_j \hat{\omega}_j = 1$ and $\sum_t \hat{\lambda}_t = 1$ — are doing all the structural work.

---

### 2. Algebraic Bias Decomposition

Under $Y_{it} = \alpha_i + 0.3\,t + \lambda_i F_t + \varepsilon_{it}$, plug the DGP into the **unit-weighted residual gap** at time $t$:

$$G_t \;\equiv\; Y_{treat,t} - \sum_j \hat{\omega}_j Y_{j,t}
\;=\; \underbrace{(\alpha_{treat} - \textstyle\sum_j \hat{\omega}_j \alpha_j)}_{\Delta\alpha} + 0.3\,t\underbrace{(1 - \textstyle\sum_j \hat{\omega}_j)}_{=\;0\ \text{by constraint}} + \underbrace{(\lambda_{treat} - \textstyle\sum_j \hat{\omega}_j \lambda_j)}_{\Delta\lambda}\,F_t + \varepsilon^*_t$$

The constraint $\sum_j \hat{\omega}_j = 1$ kills the linear time trend **exactly**, irrespective of how well the omega weights match factor loadings. This is the first unconditional layer of protection.

The gap reduces to $G_t = \Delta\alpha + \Delta\lambda\,F_t + \varepsilon^*_t$. The SDID estimand then becomes:

$$\hat{\tau}^{SDID} = \tau + \underbrace{\Delta\alpha\left(1 - \textstyle\sum_t \hat{\lambda}_t\right)}_{=\;0\ \text{by constraint}} + \Delta\lambda\!\left(\bar{F}_{post} - \sum_{t < T_0} \hat{\lambda}_t F_t\right) + \text{noise}$$

The unit-FE residual $\Delta\alpha$ also drops out because $\sum_t \hat{\lambda}_t = 1$. After both cancellations, the **entire structural bias** reduces to a single term:

$$\boxed{\;\hat{\tau}^{SDID} - \tau \;\approx\; \Delta\lambda\!\left(\bar{F}_{post} - \sum_{t < T_0} \hat{\lambda}_t F_t\right)\;}$$

---

### 3. Why the Lambda Weights Eliminate the Factor Bias

The lambda objective seeks weights that make the pre-period weighted average of *every* control match its post-period mean. Substituting the DGP into the lambda objective and ignoring noise:

$$\sum_j \left[\lambda_0 + \alpha_j\!\underbrace{(\textstyle\sum_t \lambda_t - 1)}_{=\,0} + 0.3\!\underbrace{(\textstyle\sum_t \lambda_t t - \bar{t}_{post})}_{\text{constant across }j} + \lambda_j\!\left(\textstyle\sum_t \lambda_t F_t - \bar{F}_{post}\right)\right]^2$$

The intercept $\hat{\lambda}_0$ absorbs the constant trend-deviation term. The only residual that **varies across controls** is $\lambda_j\!\left(\sum_t \lambda_t F_t - \bar{F}_{post}\right)$ — and the FOC of the sum-of-squares with respect to $\lambda$ forces this term toward zero. Specifically, differentiating in the factor direction gives:

$$\left(\textstyle\sum_t \hat{\lambda}_t F_t - \bar{F}_{post}\right) \cdot \tfrac{1}{N_{ctrl}}\!\sum_j \lambda_j^2 \;\approx\; 0$$

Since $\sum_j \lambda_j^2 > 0$, the only way to satisfy this is $\sum_t \hat{\lambda}_t F_t \approx \bar{F}_{post}$. Substituting back into the boxed bias formula:

$$\Delta\lambda\!\underbrace{\left(\bar{F}_{post} - \textstyle\sum_t \hat{\lambda}_t F_t\right)}_{\approx\,0} \;\approx\; 0 \qquad \text{for any}\ \Delta\lambda$$

**This is the key result.** Even when unit weights fail to balance factor loadings ($\Delta\lambda \neq 0$), the time weights compensate by directly interpolating the factor path. The two weight systems form a redundant pair: each can absorb confounders the other misses.

#### When the lambda cancellation can fail

The argument above requires three conditions:

1. **Feasibility of the simplex projection.** $\bar{F}_{post}$ must lie in (or near) the convex hull of $\{F_t : t < T_0\}$. If the post-period factor value is an extreme tail outside the pre-period range, no non-negative $\lambda$ with $\sum_t \lambda_t = 1$ can interpolate it exactly.
2. **Adequate pre-period length.** With $T_{pre} = 1$ there is no freedom; with $T_{pre} = 2$ only a single interpolation point. The pre-period must be long enough to span the factor's relevant variation.
3. **Identifiability** — the linear time trend and factor path must not be collinear over the pre-period (otherwise the objective cannot separately constrain $\sum_t \hat{\lambda}_t t$ and $\sum_t \hat{\lambda}_t F_t$).

In our DGP all three conditions hold with $T_{pre} \in \{10, 50\}$, which is why SDID's bias is near zero across all four cells. The small SDID variance ($\approx 0.06$–$0.08$) is dominated by the noise term $\sum_t \hat{\lambda}_t \varepsilon^*_t$, not by residual factor bias.

---

### 4. Why DiD's Failure Mode Is Variance Explosion, Not Bias

DiD imposes equal weights: $\hat{\omega}_j = 1/N$, $\hat{\lambda}_t = 1/T_{pre}$. Plugging in:

$$\hat{\tau}^{DiD} - \tau = (\lambda_{treat} - \bar{\lambda}_{ctrl})\,(\bar{F}_{post} - \bar{F}_{pre}) + \text{noise}$$

Both factors of this product are **random across DGP draws**:

| Quantity | Distribution | Independence |
|----------|--------------|--------------|
| $\lambda_{treat} - \bar{\lambda}_{ctrl}$ | $\mathcal{N}(0,\ \sigma_\lambda^2(1 + 1/N))$ | drawn from unit RNG |
| $\bar{F}_{post} - \bar{F}_{pre}$ | $\mathcal{N}(0,\ \sigma_F^2(1/T_{pre} + 1/T_{post}))$ | drawn from time RNG, ⫫ of $\lambda$'s |

So the product is mean-zero (small *Monte-Carlo* mean bias) but has variance

$$\mathrm{Var}\!\left[\hat{\tau}^{DiD}\right] \;\approx\; \sigma_\lambda^2\,\sigma_F^2\!\left(1 + \tfrac{1}{N}\right)\!\left(\tfrac{1}{T_{pre}} + \tfrac{1}{T_{post}}\right) + \text{noise variance}$$

Plugging in DGP B ($\sigma_\lambda = 0.5$, $\sigma_F = 1$, $N = 100$, $T_{pre} = 10$, $T_{post} = 3$):

$$\sqrt{0.25 \cdot 1.01 \cdot 0.433} \approx 0.33$$

The observed MC std for DiD in cell B is **0.35** — within rounding of the prediction. Cell C predicts $\sqrt{0.25 \cdot 1.05 \cdot 0.353} \approx 0.30$, observed **0.38** (the extra is from the smaller donor pool $N = 20$ amplifying noise in $\bar{Y}^{post}_{ctrl}$).

The crucial observation: **DiD's mean bias is small but its variance is structural**. A single dataset drawn from a factor DGP can produce a DiD estimate off by 0.3–0.5 in either direction even when the true ATT is 0.16, because that single draw realizes one product of two normal random variables that has no mechanism to cancel.

| Cell | $\sigma_\lambda$ | DiD mean bias | DiD std | Theoretical std |
|------|-----------------|--------------|---------|-----------------|
| Baseline | 0.0 | +0.002 | 0.058 | 0 (factor channel inactive) |
| A | 0.0 | −0.011 | 0.059 | 0 (factor channel inactive) |
| B | 0.5 | +0.041 | **0.352** | $\approx$ 0.33 ✓ |
| C | 0.5 | +0.049 | **0.383** | $\approx$ 0.30 + donor-pool noise ✓ |

---

### 5. Why SC's Bias Tracks T/N, Not $\sigma_\lambda$

SC estimates the counterfactual in **levels** with no differencing:

$$\hat{Y}^{(0)}_{treat,t} = \hat{\mu} + \sum_j \hat{\omega}_j Y_{j,t}$$

It relies entirely on $\hat{\omega}$ to handle both the time trend (needs $\sum_j \hat{\omega}_j \approx 1$) and the factor (needs $\hat{\omega}_j$ to match factor loadings).

When $T_{pre} \ll N$ the system is severely underdetermined and elastic-net regularization forces a large fraction of weights to zero: empirically $\sum_j \hat{\omega}_j \approx 0.4$–$0.7$. The shortfall propagates into the post-period extrapolation:

$$\text{bias}^{SC} \;\approx\; \underbrace{\left(1 - \textstyle\sum_j \hat{\omega}_j\right)}_{\text{shrinkage deficit}} \cdot \underbrace{0.3}_{\text{trend slope}} \cdot \underbrace{\Delta t_{post}}_{\approx\,T_{pre} + T_{post}/2}$$

For Baseline ($T_{pre}=10$, $T_{post}=3$, deficit $\approx 0.4$): $0.4 \cdot 0.3 \cdot 11 \approx +1.3$ in worst case, attenuated to roughly $+0.04$ when most of the trend is also captured by the intercept $\hat{\mu}$. The observed mean bias is $+0.037$. ✓

This is fundamentally a T/N effect, independent of $\sigma_\lambda$. Cell B (factor, T-poor) has only marginally worse RMSE (0.164) than Baseline (0.114) because the shrinkage deficit already dominates whatever extra factor-matching error appears. When T flips ($T_{pre}=50$, cells A/C) the deficit shrinks and RMSE drops by 25–40%.

---

### 6. Summary: SDID's Two Layers of Protection

| Confounding source | DiD | SC (DI) | SDID |
|---|---|---|---|
| Linear time trend $0.3\,t$ | ✓ unit differencing | ⚠ depends on $\sum\hat\omega_j \approx 1$ | ✓ unit-weight constraint $\sum_j\hat\omega_j = 1$ |
| Unit fixed effects $\Delta\alpha$ | ✓ unit differencing | ✓ intercept $\hat\mu$ | ✓ time-weight constraint $\sum_t\hat\lambda_t = 1$ |
| Factor term $\Delta\lambda \cdot F_t$ | ✗ becomes random variance | ⚠ only when T ≫ N (so $\Delta\lambda \approx 0$) | ✓ lambda weights balance $\sum_t \hat\lambda_t F_t \approx \bar F_{post}$ |
| Regularization shrinkage (T ≪ N) | n/a | ✗ levels extrapolation under-tracks | ✓ differencing absorbs residual level shift |

SDID's near-zero bias is structural, not coincidental: each of the four confounders above is killed by a specific constraint or by the lambda interpolation. DiD has only one layer (differencing) and breaks on the factor term; SC also has one layer (omega weights) and breaks when regularization is heavy. SDID combines both layers, which is why it dominates the 2×2 in cells where either single-layer estimator fails.

## Key Takeaways

### What the 2×2 reveals

| Cell | σ_λ | T/N | DiD | SC (DI) | SDID |
|------|-----|-----|-----|---------|------|
| **Baseline** | 0 | 0.1 | ✓ unbiased | ✗ upward bias (T≪N) | ✓ unbiased |
| **A: T-rich, additive** | 0 | 2.5 | ✓ unbiased | ✓ unbiased | ✓ unbiased |
| **B: T-poor, factor** | 0.5 | 0.1 | ✗ biased | ✗ biased | ~ robust |
| **C: T-rich, factor** | 0.5 | 2.5 | ✗ biased | ✓ unbiased | ✓ unbiased |

### Mechanism summary

- **DiD**: unbiased iff σ_λ = 0 (parallel trends). Factor loading dispersion directly translates to bias regardless of T/N.
- **SC (DI)**: accuracy governed by T/N. When T ≪ N, elastic-net shrinkage causes the counterfactual to under-track the post-period trend → upward bias. When T ≫ N, regularization lightens and the bias disappears. Handles factor structure well when T/N is favorable.
- **SDID**: combines SC's donor reweighting with DiD's differencing. The differencing step removes common-trend drift even when unit weights are imperfect — making it the most robust across cells. Worst case is B (factor + T≪N), where both components face stress simultaneously.